# Week 6 Spark Assignment

I've made some changes because one csv doesnt work for all the questions given  
So, either new columns are added in the csv with random but correct values  
or so questions are changed eg searching 'Technology' instead of 'Electronics' etc.

In [1]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

code needs hadoop.dll and winutils.exe to write parquets

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, sum as _sum, min, max, mean, rand, when, floor, round as _round
)

spark = SparkSession.builder.appName("sparky").getOrCreate()

df = spark.read.csv(
    "../data/superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

# type correction
df = (df
    .withColumn("Sales", col("Sales").cast("double"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Discount", col("Discount").cast("double"))
)

# Synthetic columns for fields the assignment needs
df = (df
    .withColumn("Age", (floor(rand() * 50) + 18).cast("int"))
    .withColumn("Subscription", when(rand() > 0.5, "Premium").otherwise("Standard"))
    .withColumn("Email", col("Customer Name"))
    .withColumn("Username", col("Customer ID"))
    .withColumn("status", when(rand() < 0.6, "Completed")
                          .when(rand() < 0.8, "Pending")
                          .otherwise("Cancelled"))
    .withColumn("priority", when(rand() < 0.3, "High")
                            .when(rand() < 0.7, "Medium")
                            .otherwise("Low"))
    .withColumn("base_price", _round(col("Sales") / (col("Quantity") + 0.0001), 2))
    .withColumn("user_id", col("Customer ID"))
)

df.printSchema()
df.show(2,vertical=True)

Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.  

Driver: The process running your main()/script. It builds the DAG of transformations, creates the SparkContext, negotiates resources with the Cluster Manager, and schedules tasks onto executors. It also collects results back (for actions like collect()).
Cluster Manager: Allocates resources across the cluster. Could be YARN, Kubernetes, Mesos, or Spark's own Standalone manager. It doesn't run your Spark logic just hands out worker nodes/containers to the Driver.
Executor: JVM processes on worker nodes that actually run tasks and hold data in memory/disk (as partitions). Each executor runs multiple tasks in parallel threads and reports status back to the Driver.

Flow: Driver → asks Cluster Manager for resources → Cluster Manager launches Executors → Driver sends tasks to Executors → Executors process data and return results/status.


Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?  

Spark doesn't execute transformations (filter, select, map, etc.) immediately, it just builds a logical plan (DAG) of what needs to happen. Execution only triggers when an action (show, count, write, collect) is called.

Why it helps performance:

Spark's Catalyst optimizer can look at the entire chain before running anything, and reorder/merge operations (e.g., push a filter before a join, combine multiple select+filter into one pass instead of materializing intermediate results).
Avoids wasted computation — if you filter down to 1% of data before a heavy groupBy, Spark can apply that filter early instead of processing everything then discarding.
No unnecessary intermediate DataFrames get written to memory/disk.

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

already read at the beginning
df = spark.read.csv("../data/superstore.csv", header=True, inferSchema=True, multiLine=True, escape='"')

Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

CSV: row-based, plain text. To read even one column, Spark has to read every byte of every row.
Parquet: columnar, binary, compressed, stores schema + stats (min/max per column) in metadata.
Reading fewer columns from Parquet = actual less I/O (column pruning). CSV gives you no such benefit, you always read the whole row.
Parquet supports predicate pushdown (see Q9) and compresses far better (repeated values in a column compress well; mixed row data doesn't).
CSV has no schema, everything comes in as string unless you infer, which is slow and error-prone (inferSchema=true triggers an extra full-data scan).


Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.  

used Technology instead

In [ ]:
df.select("Product ID", "base_price").filter(col("Category") == "Technology").show(5)

Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [ ]:
df_revised = (df
    .withColumnRenamed("Sales", "revenue")
    .withColumn("base_price", col("base_price").cast("double"))
)
df_revised.printSchema()

Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?   

Spark tracks every transformation applied to a base RDD/DataFrame as a lineage (DAG), not the data itself, but the recipe to produce it. If a worker/executor dies mid-job and loses a partition, Spark doesn't need a replica of that data sitting somewhere, it just recomputes the lost partition by replaying the lineage from the original source (or last checkpoint). This is why Spark doesn't need synchronous replication like a traditional distributed database, recomputation from lineage is cheaper than always replicating.


Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [ ]:
df.filter((col("status") == "Completed") & (col("base_price") > 1000)).show(2,vertical=True)

Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. 
  
Predicate pushdown means filter conditions in your query get pushed down to the storage layer (Parquet reader) instead of being applied after all data is loaded into memory. Parquet stores per-column-chunk statistics (min/max, null counts) in its footer metadata. When you do df.filter(col("amount") > 1000), Spark checks those stats first and can skip entire row groups that can't possibly satisfy the condition without decompressing or reading them at all. Net effect: less data physically read off disk and less data deserialized into memory, which cuts I/O and speeds up the query significantly, especially on large files.

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [ ]:
df = df.withColumn("final_price", _round(col("base_price") * 1.18, 2))
df.select("base_price", "final_price").show(5)

Q11: What is the difference between Transformations and Actions? Provide two examples of each.

Transformations: lazy, return a new DataFrame/RDD, don't trigger execution. Examples: filter(), select(), withColumn(), groupBy(), join().
Actions: trigger actual execution of the DAG built so far, return a result to the Driver or write output. Examples: count(), show(), collect(), write.parquet().

Rule of thumb: if it returns a DataFrame, it's a transformation; if it returns a value/writes output/prints something, it's an action.


Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

In [ ]:
df.write.mode("overwrite").parquet("../data/superstore_parquet")

df_parquet = spark.read.parquet("../data/superstore_parquet")
df_clean = df_parquet.filter(col("user_id").isNotNull())
df_clean.write.mode("overwrite").csv("../data/superstore_output_csv", header=True)


Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

Client Mode: the Driver runs on the machine where you launched the job (your laptop/edge node), while executors run on the cluster. If your machine disconnects or crashes, the job dies. Common for interactive work (spark-shell, notebooks, debugging).
Cluster Mode: the Driver itself runs inside the cluster (on one of the worker nodes), managed by the Cluster Manager. You can disconnect your terminal and the job keeps running. Standard for production jobs submitted via spark-submit.


Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 

In [ ]:
df.select("Region").distinct().show() #to show available regions

using region West instead of North

In [ ]:
df.filter((col("Region") == "West") | (col("priority") == "High")).show(2,vertical=True)


Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?  

.collect() pulls every row of the result set back to the Driver's memory as a local Python/Scala list. On a multi-terabyte dataset, that will either take forever or straight-up crash the Driver with an OOM the Driver is a single JVM, not a distributed store. .show(5) only computes and materializes 5 rows (Spark is smart enough to limit the actual work done, not just the display), so it's a cheap, safe way to peek at data without ever risking Driver memory.

In [ ]:
df.show(5,vertical=True)